In [1]:
!pip install -q langchain langchain-groq gspread

In [11]:
import os
from google.colab import userdata, auth

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=3
)

# Google Sheets Authentication
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Create or open Google Sheet
try:
    sh = gc.open("ShopMate AI Orders")
except gspread.SpreadsheetNotFound:
    sh = gc.create("ShopMate AI Orders")

ws = sh.sheet1

# Add headers only once
if not ws.get_all_values():
    ws.append_row([
        "name",
        "item",
        "qty",
        "size",
        "color",
        "date",
        "total"
    ])

print("Sheet ready:", sh.url)

Sheet ready: https://docs.google.com/spreadsheets/d/1manRtlmq51VLZAHVTQosLSnOMNSi6AxAYxCS81HyJoI


In [12]:
from typing import Literal
from pydantic import BaseModel, Field

from langchain.tools import tool

PRODUCTS = {
    "kurti": {
        "price": 599,
        "colors": ["Black", "Pink", "Blue"],
        "sizes": ["S", "M", "L", "XL"]
    },

    "anarkali dress": {
        "price": 899,
        "colors": ["Red", "Blue", "Pink"],
        "sizes": ["S", "M", "L", "XL"]
    },

    "saree": {
        "price": 1299,
        "colors": ["Red", "Green", "Blue"],
        "sizes": ["Free Size"]
    },

    "jeans": {
        "price": 799,
        "colors": ["Black", "Blue"],
        "sizes": ["28", "30", "32", "34"]
    },

    "top": {
        "price": 499,
        "colors": ["White", "Black", "Pink"],
        "sizes": ["S", "M", "L", "XL"]
    },

    "handbag": {
        "price": 699,
        "colors": ["Brown", "Black", "Cream"],
        "sizes": ["One Size"]
    }
}

In [13]:
class Order(BaseModel):
    """A fashion shopping order taken from a customer's chat."""

    name: str = Field(
        description="Customer's first name"
    )

    item: Literal[
        "kurti",
        "anarkali dress",
        "saree",
        "jeans",
        "top",
        "handbag"
    ] = Field(
        description="Selected product"
    )

    qty: int = Field(
        description="Number of products",
        ge=1
    )

    size: str = Field(
        description="Selected product size"
    )

    color: str = Field(
        description="Selected product colour"
    )

    date: str = Field(
        description="Delivery or pickup date as the customer said it"
    )

In [14]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price for a fashion product.

    Args:
        item: Product name, e.g. kurti, saree, jeans.
        qty: Number of products.
    """

    product = PRODUCTS.get(item.lower().strip())

    if product is None:
        return f"'{item}' is unavailable in our catalog."

    if qty < 1:
        return "Quantity must be at least 1."

    price = product["price"]
    total = price * qty

    return f"{qty} x {item} = Rs {total}"

In [15]:
@tool
def check_product(item: str) -> str:
    """Check product details, available colours, sizes and price.

    Args:
        item: Name of the fashion product.
    """

    product = PRODUCTS.get(item.lower().strip())

    if product is None:
        return f"'{item}' is not available. Products: {', '.join(PRODUCTS)}."

    return f"""
Product: {item}
Price: Rs {product['price']}
Available Colours: {', '.join(product['colors'])}
Available Sizes: {', '.join(product['sizes'])}
"""

In [16]:
@tool
def recommend_products(budget: int) -> str:
    """Recommend fashion products within the customer's budget.

    Args:
        budget: Maximum shopping budget in rupees.
    """

    if budget < 0:
        return "Please provide a valid budget."

    matches = []

    for item, details in PRODUCTS.items():
        if details["price"] <= budget:
            matches.append(
                f"{item} - Rs {details['price']}"
            )

    if not matches:
        return "No products are available within this budget."

    return "Products within your budget:\n" + "\n".join(matches)

In [17]:
@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    size: str,
    color: str,
    date: str
) -> str:
    """Save a confirmed fashion shopping order to Google Sheets.

    Use only after the customer has explicitly confirmed
    the order and provided all required details.

    Args:
        name: Customer name.
        item: Selected product.
        qty: Quantity.
        size: Selected size.
        color: Selected colour.
        date: Delivery or pickup date.
    """

    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            size=size.strip().upper(),
            color=color.strip().title(),
            date=date
        )

    except Exception as e:
        return f"Could not save order: {e}"

    product = PRODUCTS[order.item]

    if order.color.lower() not in [
        c.lower() for c in product["colors"]
    ]:
        return f"Colour {order.color} is unavailable."

    if order.size.lower() not in [
        s.lower() for s in product["sizes"]
    ]:
        return f"Size {order.size} is unavailable."

    total = product["price"] * order.qty

    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.size,
        order.color,
        order.date,
        total
    ])

    return f"""
Order saved successfully!

Customer: {order.name}
Product: {order.item}
Quantity: {order.qty}
Size: {order.size}
Colour: {order.color}
Date: {order.date}
Total Amount: Rs {total}
"""

In [18]:
from langchain.agents import create_agent

SYSTEM = """
ROLE:
You are ShopMate AI, a friendly and intelligent fashion shopping assistant.

FACTS:
You help customers discover fashion products, compare prices,
check sizes, choose colours and place shopping orders.

PRODUCT CATALOG:
Use the available product catalog and tools for accurate information.

RULES:
1. Always use get_price for product prices.
2. Always use check_product for available colours and sizes.
3. Use recommend_products when customers mention their budget.
4. Never invent products, prices, colours or sizes.
5. Ask customers for missing order details.
6. Collect customer name, product, quantity, size, colour and date.
7. Before calling log_order, summarize the complete order
   and ask the customer for confirmation.
8. Call log_order only after explicit customer confirmation.
9. After saving, read the complete order back with its total.
10. Never claim an order is saved unless the tool confirms it.

LANGUAGE:
Support English and Telugu.
Respond in the same language used by the customer.

FORMAT:
Keep responses short, friendly and professional.
Use simple language.
"""

agent = create_agent(
    model=llm,
    tools=[
        get_price,
        check_product,
        recommend_products,
        log_order
    ],
    system_prompt=SYSTEM
)

In [19]:
history = []

def chat(text: str) -> str:

    global history

    try:
        result = agent.invoke({
            "messages": history + [
                {"role": "user", "content": text}
            ]
        })

        history = result["messages"]

        return history[-1].content

    except Exception as e:
        print("Agent error:", e)

        return "Sorry, I'm a little busy. Please try again."

In [20]:
print(chat("Hi, what fashion products are available?"))

Hi! 👋  
We have a variety of items such as kurtis, sarees, jeans, dresses, tops, and more.  
Could you let me know what type of product you’re looking for? That way I can give you the exact options, colours, sizes, and prices.


In [21]:
print(chat("What is the price of 2 kurtis?"))

The price for 2 kurtis is **₹1,198**.  
Would you like to place an order, or do you need more details (colour, size, etc.)?


In [22]:
print(chat("Na budget 1000 rupees. Manchi dresses suggest cheyyi."))

మీ బడ్జెట్ ₹1000 లోపు ఉన్న కొన్ని మంచి డ్రెస్‌లు ఇవి:

| ఉత్పత్తి | ధర |
|----------|-----|
| **Kurti** | ₹599 |
| **Anarkali Dress** | ₹899 |
| **Jeans** | ₹799 |
| **Top** | ₹499 |
| **Handbag** | ₹699 |

మీకు ఏదైనా ప్రత్యేకంగా ఆసక్తి ఉంటే, రంగు, సైజు, పరిమాణం, డెలివరీ తేదీ వంటి వివరాలు చెబితే ఆర్డర్ పెట్టడంలో సహాయం చేస్తాను.


In [23]:
print(chat(
    "My name is Sree. I want 2 kurtis, "
    "size M, black colour, delivery on 28 November."
))

Hi Sree!  
Here’s your order summary:

- **Product:** Kurti  
- **Quantity:** 2  
- **Size:** M  
- **Colour:** Black  
- **Delivery date:** 28 November  

Total cost: ₹1,198  

Is everything correct? If yes, just say “yes” and I’ll save the order.


In [24]:
print("ShopMate AI Orders:")
print(sh.url)

orders = ws.get_all_records()

for order in orders:
    print(order)

ShopMate AI Orders:
https://docs.google.com/spreadsheets/d/1manRtlmq51VLZAHVTQosLSnOMNSi6AxAYxCS81HyJoI
